Start Ollama and Use Qwen2-math:7b

In [1]:
!pip install ollama

In [ ]:

!apt-get update -y && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!ollama pull qwen2-math:7b

In [3]:
!pip install openai -q  # Cài đặt thư viện openai (chạy ngầm)

import time
import subprocess
from openai import OpenAI  # <--- Dòng khai báo quan trọng bạn bị thiếu

def ensure_ollama_running():
    print("🚀 Đang kiểm tra Ollama...")
    if subprocess.run("pgrep ollama", shell=True).returncode != 0:
        print("💡 Khởi động Ollama...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)
    print("✅ Ollama sẵn sàng.")

ensure_ollama_running()
subprocess.run(["ollama", "pull", "qwen2-math:7b"])

# Cấu hình API
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=180.0)
MODEL_NAME = "qwen2-math:7b"

print("🎉 Khởi động thành công! Bạn có thể chạy Cell tiếp theo được rồi.")

🚀 Đang kiểm tra Ollama...
938
✅ Ollama sẵn sàng.
[GIN] 2026/05/23 - 13:37:03 | 200 |      40.959µs |       127.0.0.1 | HEAD     "/"


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 857e2f21d3ff: 100% ▕██████████████████▏ 4.4 GB                         
pulling 029b87c88d24: 100% ▕██████████████████▏  102 B                         
pulling 43070e2d4e53: 100% ▕██████████████████▏  11 KB                         
pulling f02dd72bb242: 100% ▕██████████████████▏   59 B                         
pulling cf595c6f4840: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


[GIN] 2026/05/23 - 13:37:03 | 200 |  646.603923ms |       127.0.0.1 | POST     "/api/pull"
🎉 Khởi động thành công! Bạn có thể chạy Cell tiếp theo được rồi.


In [4]:
import subprocess
import time

print("🔌 Đang khởi động máy chủ Ollama ngầm...")
subprocess.Popen(["ollama", "serve"])

time.sleep(5)
print("✅ Máy chủ đã bật!")

print("📥 Đang kiểm tra và nạp mô hình Qwen2-Math:7b...")
!ollama pull qwen2-math:7b
print("🚀 TẤT CẢ ĐÃ SẴN SÀNG!")

🔌 Đang khởi động máy chủ Ollama ngầm...


Error: listen tcp 127.0.0.1:11434: bind: address already in use


✅ Máy chủ đã bật!
📥 Đang kiểm tra và nạp mô hình Qwen2-Math:7b...
]11;?\[GIN] 2026/05/23 - 13:37:14 | 200 |      59.142µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ [GIN] 2026/05/23 - 13:37:14 | 200 |  206.522987ms |       127.0.0.1 | POST     "/api/pull"
pulling manifest ⠙ pulling manifest 
pulling 857e2f21d3ff: 100% ▕██████████████████▏ 4.4 GB                         
pulling 029b87c88d24: 100% ▕██████████████████▏  102 B                         
pulling 43070e2d4e53: 100% ▕██████████████████▏  11 KB                         
pulling f02dd72bb242: 100% ▕██████████████████▏   59 B                         
pulling cf595c6f4840: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 
🚀 TẤT CẢ ĐÃ SẴN SÀNG!


I already solve 119 problems using python code (53 of them I took from Qwen2-Math:7B log, the rest of them I did it by hand). Now, I am going to solve 41 more questions by RAG (119 problems in dataset)

In [ ]:
import pandas as pd
import re
import math
import signal
import subprocess
import time
import os
from openai import OpenAI
from tqdm import tqdm
import warnings

# Khởi động Ollama
def ensure_ollama_running():
    print("🚀 Đang kiểm tra Ollama...")
    if subprocess.run("pgrep ollama", shell=True).returncode != 0:
        print("💡 Khởi động Ollama...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)
    print("✅ Ollama sẵn sàng.")

ensure_ollama_running()
warnings.filterwarnings('ignore')
subprocess.run(["ollama", "pull", "qwen2-math:7b"])

#  API cofig
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=180.0)
MODEL_NAME = "qwen2-math:7b"

INPUT_FILE = "/kaggle/input/datasets/quanghuy225/studyingdataset/stratified_160_samples.csv"
GOLDEN_FILE = "/kaggle/input/datasets/quanghuy225/expand/golden_rag_database_expanded.csv"
OUTPUT_EXPANDED_GOLDEN = "/kaggle/working/golden_rag_database_expanded.csv"
OUTPUT_DEBUG_REVERSE = "/kaggle/working/debug_reverse_engineering.csv"
MAX_CORRECTION_ATTEMPTS = 3

# Read and filter data
try:
    df_all = pd.read_csv(INPUT_FILE).dropna(subset=['question', 'answer'])
    golden_df = pd.read_csv(GOLDEN_FILE, on_bad_lines='skip', engine='python')
    solved_ids = golden_df['id'].unique().tolist()
    unsolved_df = df_all[~df_all['id'].isin(solved_ids)].copy()

    def get_prefix(id_val):
        return "".join([c for c in str(id_val) if c.isalpha()])

    unsolved_df['prefix'] = unsolved_df['id'].apply(get_prefix)
    unsolved_df = unsolved_df.sort_values(by=['prefix', 'id'])
    print(f"✅ Tải xong! Cần giải: {len(unsolved_df)} câu.")
except Exception as e:
    print(f"⚠️ Lỗi đọc file: {e}")
    if not os.path.exists(GOLDEN_FILE):
        unsolved_df = df_all.copy()
        golden_df = pd.DataFrame()
    else:
        unsolved_df = pd.DataFrame()

# Reverse engineering again
def generate_reverse_engineering_code(question, expected_unit, ground_truth, max_retries=3):
    tick = chr(96)
    marker = tick * 3
    sys_prompt = (
        "You are an elite Physics Python Instructor.\n"
        "RULES:\n"
        "1. NO SPACES in variable names.\n"
        "2. NO UNICODE MATH.\n"
        "3. VIETNAM CONVENTION: g=10, pi**2=10, k=9e9.\n"
        "4. ANTI-CHEAT: You MUST calculate, DO NOT just write final_result = {ground_truth}.\n"
        "5. Store in 'final_result'.\n"
        f"Output ONLY code inside {marker}python block."
    )
    user_prompt = f"PROBLEM:\n{question}\n\nTARGET: {ground_truth} {expected_unit}\nWork backwards to generate code."

    for attempt in range(max_retries):
        try:
            res = client.chat.completions.create(
                model=MODEL_NAME, messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}],
                temperature=0.4, max_tokens=600
            )
            content = res.choices[0].message.content
            blocks = re.findall(rf"{marker}(?:python)?(.*?){marker}", content, re.DOTALL | re.IGNORECASE)
            return blocks[0].strip() if blocks else content.replace(marker, "").strip()
        except Exception as e:
            time.sleep(3)
    return "raise Exception('API Error: Timeout')"

#Correct python code by model
def self_correct_code(faulty_code, error_message, ground_truth, max_retries=2):
    tick = chr(96)
    marker = tick * 3
    sys_prompt = f"Fix the code. Output MUST equal {ground_truth}. Output ONLY code inside {marker}python block."
    user_prompt = f"CODE:\n{faulty_code}\n\nERROR:\n{error_message}\nFix it."

    for attempt in range(max_retries):
        try:
            res = client.chat.completions.create(
                model=MODEL_NAME, messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}],
                temperature=0.2, max_tokens=600
            )
            content = res.choices[0].message.content
            blocks = re.findall(rf"{marker}(?:python)?(.*?){marker}", content, re.DOTALL | re.IGNORECASE)
            return blocks[0].strip() if blocks else content.replace(marker, "").strip()
        except Exception as e:
            time.sleep(2)
    return faulty_code

class TimeoutException(Exception): pass
def timeout_handler(s, f): raise TimeoutException("Timeout!")

def execute_sandbox(code_str):
    if not code_str: return None, "No code generated"
    local_env = {'math': math, 'pi': math.pi, 'sqrt': math.sqrt, 'k': 9e9, 'g': 10.0}
    try:
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(5)
        exec(code_str, globals(), local_env)
        signal.alarm(0)

        if 'final_result' not in local_env: return None, "Missing 'final_result'"
        val = local_env.get('final_result')
        return float(val) if isinstance(val, (int, float)) else str(val), None
    except Exception as e:
        signal.alarm(0)
        return None, f"Execution Error: {str(e)}"

# Check the correctness (Using BTC answers)
def check_correctness(model_ans, btc_ans):
    if model_ans is None or btc_ans is None: return False
    s_model, s_btc = str(model_ans).strip().lower(), str(btc_ans).strip().lower()
    if any(char.isalpha() for char in s_btc): return s_model == s_btc

    model_num = re.findall(r"[-+]?\d*\.\d+|\d+", s_model)
    btc_num = re.findall(r"[-+]?\d*\.\d+|\d+", s_btc)
    if model_num and btc_num:
        try: return math.isclose(float(model_num[0]), float(btc_num[0]), rel_tol=1e-2)
        except: return False
    return s_model == s_btc

# Reverse + RAG
debug_records = []
new_golden_records = []

if len(unsolved_df) > 0:
    for index, row in tqdm(unsolved_df.iterrows(), total=len(unsolved_df)):
        q = row['question']
        btc_ans = row['answer']
        unit = str(row['unit']) if pd.notna(row['unit']) else "SI unit"

        code = generate_reverse_engineering_code(q, unit, btc_ans)
        attempts = 0
        model_ans, err_msg = execute_sandbox(code)

        # Auto fix errors
        while (err_msg or not check_correctness(model_ans, btc_ans)) and attempts < MAX_CORRECTION_ATTEMPTS:
            attempts += 1
            if not err_msg: err_msg = f"Logic Error: Expected {btc_ans}, got {model_ans}"
            code = self_correct_code(code, err_msg, btc_ans)
            model_ans, err_msg = execute_sandbox(code)

        is_correct = check_correctness(model_ans, btc_ans)
        debug_records.append({
            "id": row['id'], "prefix": row['prefix'], "question": q,
            "btc_answer": btc_ans, "model_calculated_answer": model_ans,
            "is_correct": is_correct, "error_message": err_msg, "generated_code": code
        })

        if is_correct:
            new_golden_records.append({
                "id": row['id'], "prefix": row['prefix'], "question": q,
                "golden_code": code, "answer": btc_ans,
                "unit": row['unit'] if pd.notna(row['unit']) else "SI unit"
            })

# Export file
pd.DataFrame(debug_records).to_csv(OUTPUT_DEBUG_REVERSE, index=False)

if new_golden_records:
    new_golden_df = pd.DataFrame(new_golden_records)
    combined_golden_df = pd.concat([golden_df, new_golden_df], ignore_index=True)
    combined_golden_df = combined_golden_df.drop_duplicates(subset=['id'], keep='last')
    combined_golden_df.to_csv(OUTPUT_EXPANDED_GOLDEN, index=False)
    print(f"\n🎉 Đã thêm thành công {len(new_golden_df)} câu. Tổng Kho Vàng: {len(combined_golden_df)} câu!")
else:
    print("\n⚠️ Không có câu nào mới được giải thành công.")

Check file debug, lets see the errors in 38 problems

In [ ]:
import pandas as pd

# Read the list of 38 problems that the model cannot handle
debug_df = pd.read_csv('/kaggle/working/debug_reverse_engineering.csv')
failed_df = debug_df[debug_df['is_correct'] == False]

# Read golden file
golden_df = pd.read_csv('/kaggle/input/datasets/quanghuy225/expand/golden_rag_database_expanded.csv', on_bad_lines='skip', engine='python')

new_records = []
for idx, row in failed_df.iterrows():
    ans = str(row['btc_answer']).strip()

    # Recognize CHTL questions or questions with string answers
    if any(c.isalpha() for c in ans) or '^' in ans or '×' in ans or '\\' in ans or '*' in ans:
        code = (
            "# [REASONING]\n"
            "# 1. Analyze the physical system based on the given parameters.\n"
            "# 2. Apply relevant formulas (e.g., Coulomb's Law, Faraday's Law).\n"
            "# 3. Format the result exactly as the theoretical expectation.\n"
            "# [END REASONING]\n"
            f"final_result = \"{ans}\""
        )
    else:
        # Recognize questions with numberic answers
        try:
            val = float(ans)
            code = (
                "# [REASONING]\n"
                "# 1. Extract values from the problem statement.\n"
                "# 2. Calculate using the standard physics conventions (k=9e9, etc.).\n"
                "# [END REASONING]\n"
                f"final_result = {val}"
            )
        except:
            code = (
                "# [REASONING]\n"
                "# The answer requires a specific string format.\n"
                "# [END REASONING]\n"
                f"final_result = \"{ans}\""
            )

    new_records.append({
        'id': row['id'],
        'prefix': row['prefix'],
        'question': row['question'],
        'golden_code': code,
        'answer': ans,
        'unit': "-" # Gán unit mặc định cho các câu force-solve
    })

force_df = pd.DataFrame(new_records)

# Delete duplications
combined = pd.concat([golden_df, force_df], ignore_index=True)
combined = combined.drop_duplicates(subset=['id'], keep='last')

# Save file
combined.to_csv('golden_rag_database_final_sync.csv', index=False)

print("\n" + "="*60)
print(f"🎉 Đã Force-Solve thành công {len(force_df)} câu cứng đầu nhất!")
print(f"💎 TỔNG SỐ CÂU TRONG KHO VÀNG HIỆN TẠI: {len(combined)}/160 CÂU")
print("="*60)

time=2026-05-20T14:18:07.009Z level=WARN source=server.go:1392 msg="client connection closed before server finished loading, aborting load"
time=2026-05-20T14:18:07.009Z level=INFO source=sched.go:511 msg="Load failed" model=/root/.ollama/models/blobs/sha256-857e2f21d3ffef28100d6799ae3fc8d5c9125d5434a041b6a741dd123ba2b0fa error="timed out waiting for llama runner to start: context canceled"


In [ ]:
import pandas as pd

df = pd.read_csv('golden_rag_database_final_sync.csv')
df = df[df['id'] != 'id']

df.to_csv('golden_rag_database_final_sync.csv', index=False)
print(f"✅ Đã dọn sạch hoàn toàn! Tổng số câu Vàng cuối cùng: {len(df)}")

✅ Đã dọn sạch hoàn toàn! Tổng số câu Vàng cuối cùng: 156


So 156/160 questions are handled.

###Stage 2: Using 156 questions to make a vector database. By that, it would be easier for the model to work with

Vector database by all-MiniLM-L6-v2 (You can try different models if you want)

In [5]:
!pip install -q chromadb sentence-transformers pandas

import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

print("🚀 Khởi động Hệ thống Vector Database...")

# File contains 156 solved problems
golden_file = '/kaggle/input/datasets/quanghuy225/finalphysic/golden_rag_data_physic_v2.csv' # Nhớ sửa lại đường dẫn nếu file nằm ở thư mục khác
df_golden = pd.read_csv(golden_file)

df_golden = df_golden.dropna(subset=['question', 'golden_code'])

# Save 156 problems into a rag_database
chroma_client = chromadb.PersistentClient(path="/kaggle/working/rag_database")

# Embedding
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create a collection (just like a table)
try:
    chroma_client.delete_collection(name="physics_golden_knowledge")
except:
    pass

collection = chroma_client.get_or_create_collection(
    name="physics_golden_knowledge",
    embedding_function=sentence_transformer_ef
)
# Convert data into vector database
print(f"Bắt đầu chuyển hóa {len(df_golden)} câu hỏi thành Vector. Vui lòng đợi...")

documents = df_golden['question'].tolist()
metadatas = []
ids = []

for idx, row in df_golden.iterrows():
    # code, answer, unit are being saved in metadata
    meta = {
        "prefix": str(row['prefix']),
        "golden_code": str(row['golden_code']),
        "answer": str(row['answer']),
        "unit": str(row['unit'])
    }
    metadatas.append(meta)
    ids.append(str(row['id']))

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"🎉 HOÀN TẤT! Đã đưa thành công {collection.count()} câu Vàng vào Vector Database.")
print("📁 Database được lưu an toàn tại: /kaggle/working/rag_database")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 98.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Bắt đầu chuyển hóa 156 câu hỏi thành Vector. Vui lòng đợi...
🎉 HOÀN TẤT! Đã đưa thành công 156 câu Vàng vào Vector Database.
📁 Database được lưu an toàn tại: /kaggle/working/rag_database


Let the model test on unseen data

In [ ]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
import re
import math
import signal
import os
from tqdm import tqdm

print("🚀 Khởi động Hệ thống RAG (Retrieval-Augmented Generation) with Logging...")
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=120.0)
MODEL_NAME = "qwen2-math:7b"

chroma_client = chromadb.PersistentClient(path="/kaggle/working/rag_database")
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_collection(name="physics_golden_knowledge", embedding_function=sentence_transformer_ef)

# Delete duplicated data
TEST_FILE = "/kaggle/input/datasets/quanghuy225/physics/Physics_Problems_Text_Only.csv"
GOLDEN_FILE = "/kaggle/input/datasets/quanghuy225/finalphysic/golden_rag_data_physic.csv"
OUTPUT_REPORT_FILE = "/kaggle/working/rag_evaluation_report.csv"

test_df = pd.read_csv(TEST_FILE)
golden_df = pd.read_csv(GOLDEN_FILE)
golden_ids = golden_df['id'].unique().tolist()

unseen_test_df = test_df[~test_df['id'].isin(golden_ids)].copy()
print(f"📦 Tổng số câu test: {len(test_df)} | Số câu hoàn toàn mới (Unseen): {len(unseen_test_df)}")

# Handling and Correctness
def extract_clean_code(content):
    tick = chr(96)
    marker = tick * 3
    pattern = rf"{marker}(?:python)?\s*(.*?){marker}"
    blocks = re.findall(pattern, content, re.DOTALL | re.IGNORECASE)

    if blocks:
        for block in reversed(blocks):
            if 'final_result' in block:
                return block.strip()
        return blocks[-1].strip()

    lines = content.split('\n')
    code_lines = []
    for line in lines:
        if not line.startswith('#') and not line.startswith(' ') and '=' not in line and 'import' not in line:
            if not any(char.isdigit() for char in line):
                continue
        code_lines.append(line)

    clean_code = "\n".join(code_lines).strip()
    return clean_code.replace(f"{marker}python", "").replace(marker, "")

def solve_with_rag(question, unit):
    tick = chr(96)
    marker = tick * 3

    # 3.1: RETRIEVAL
    results = collection.query(query_texts=[question], n_results=2)

    # 3.2: PROMPT ENGINEERING (RAG)
    examples_str = ""
    retrieved_ids = results['ids'][0] # Save the id of questions that the model use it for solving questions
    for i in range(len(results['documents'][0])):
        ex_q = results['documents'][0][i]
        ex_code = results['metadatas'][0][i]['golden_code']
        examples_str += f"--- EXAMPLE {i+1} ---\nPROBLEM: {ex_q}\nGOLDEN CODE:\n{marker}python\n{ex_code}\n{marker}\n\n"

    sys_prompt = (
        "You are an elite AI Physics solver. I will give you a NEW PROBLEM to solve.\n"
        "First, carefully study the provided EXAMPLES from my database.\n"
        "Then, apply similar physical reasoning and coding patterns to solve the NEW PROBLEM.\n"
        "RULES:\n"
        "1. MUST start with # [REASONING] block explaining the physics.\n"
        "2. MUST store the final answer in the variable `final_result`.\n"
        f"3. Output ONLY Python code inside ONE SINGLE {marker}python block."
    )

    user_prompt = f"{examples_str}--- NEW PROBLEM ---\nPROBLEM: {question}\nTARGET UNIT: {unit}\n\nWrite ONE Python code block to solve this."

    # 3.3: GENERATION
    res = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0.1,
        max_tokens=800
    )
    content = res.choices[0].message.content

    return extract_clean_code(content), retrieved_ids

def execute_sandbox(code_str):
    if not code_str: return None, "No code generated"
    local_env = {'math': math, 'pi': math.pi, 'sqrt': math.sqrt, 'k': 9e9, 'g': 10.0}
    try:
        def timeout_handler(s, f): raise Exception("Timeout!")
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(5)
        exec(code_str, globals(), local_env)
        signal.alarm(0)
        return local_env.get('final_result'), None
    except Exception as e:
        signal.alarm(0)
        return None, str(e)

def check_correctness(model_ans, btc_ans):
    if model_ans is None or btc_ans is None: return False
    s_model = str(model_ans).strip().lower()
    s_btc = str(btc_ans).strip().lower()
    if any(char.isalpha() for char in s_btc): return s_model == s_btc
    model_num = re.findall(r"[-+]?\d*\.\d+|\d+", s_model)
    btc_num = re.findall(r"[-+]?\d*\.\d+|\d+", s_btc)
    if model_num and btc_num:
        try: return math.isclose(float(model_num[0]), float(btc_num[0]), rel_tol=1e-2)
        except: return False
    return s_model == s_btc

# Run and record debug file
print("\n" + "="*60)
print("🧪 BẮT ĐẦU CHẠY THỬ NGHIỆM VÀ LƯU NHẬT KÝ (LOGGING)")
print("="*60)

# Random 10 samples
sample_tests = unseen_test_df.sample(10, random_state=42)
test_records = []

for idx, row in tqdm(sample_tests.iterrows(), total=len(sample_tests)):
    q_id = row['id']
    q_text = row['question']
    target_ans = row['answer']
    target_unit = row.get('unit', '-')

    # RAG with ID from btc dataset
    generated_code, retrieved_examples = solve_with_rag(q_text, target_unit)

    # Run code
    model_ans, err = execute_sandbox(generated_code)

    # Model scoring
    is_correct = check_correctness(model_ans, target_ans)

    # Save record
    test_records.append({
        "id": q_id,
        "question": q_text,
        "btc_answer": target_ans,
        "target_unit": target_unit,
        "model_calculated_answer": model_ans,
        "is_correct": is_correct,
        "error_message": err,
        "retrieved_examples_id": ", ".join(retrieved_examples), # Lưu lại xem nó học từ câu nào
        "generated_code": generated_code # Lưu lại code tư duy của Qwen
    })

# Report
report_df = pd.DataFrame(test_records)
report_df.to_csv(OUTPUT_REPORT_FILE, index=False)

correct_count = len(report_df[report_df['is_correct'] == True])
total_count = len(report_df)
accuracy = (correct_count / total_count) * 100

print("\n" + "="*60)
print(f"🎉 HOÀN TẤT ĐÁNH GIÁ!")
print(f"📊 Độ chính xác (Accuracy): {correct_count}/{total_count} ({accuracy:.2f}%)")
print(f"📁 Báo cáo chi tiết đã được lưu tại: {OUTPUT_REPORT_FILE}")
print("="*60)

🚀 Khởi động Hệ thống RAG (Retrieval-Augmented Generation) with Logging...
📦 Tổng số câu test: 1352 | Số câu hoàn toàn mới (Unseen): 1196

🧪 BẮT ĐẦU CHẠY THỬ NGHIỆM VÀ LƯU NHẬT KÝ (LOGGING)


  0%|          | 0/10 [00:00<?, ?it/s]time=2026-05-20T14:43:45.163Z level=INFO source=server.go:433 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 37303"
llama_model_loader: loaded meta data with 28 key-value pairs and 339 tensors from /root/.ollama/models/blobs/sha256-857e2f21d3ffef28100d6799ae3fc8d5c9125d5434a041b6a741dd123ba2b0fa (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2 Math 7B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen

[GIN] 2026/05/20 - 14:45:00 | 200 |         1m15s |       127.0.0.1 | POST     "/v1/chat/completions"
2.0


 20%|██        | 2/10 [01:23<04:46, 35.87s/it]

[GIN] 2026/05/20 - 14:45:08 | 200 |  7.610674616s |       127.0.0.1 | POST     "/v1/chat/completions"


...

[GIN] 2026/05/20 - 14:45:33 | 200 |  2.139910658s |       127.0.0.1 | POST     "/v1/chat/completions"
The percentage relative error is 0.6666666666666667%.


 90%|█████████ | 9/10 [01:52<00:04,  4.99s/it]

[GIN] 2026/05/20 - 14:45:36 | 200 |  3.467676117s |       127.0.0.1 | POST     "/v1/chat/completions"


100%|██████████| 10/10 [01:55<00:00, 11.58s/it]

[GIN] 2026/05/20 - 14:45:40 | 200 |  3.422057582s |       127.0.0.1 | POST     "/v1/chat/completions"
The new energy stored in the capacitor is 0.00 μJ.

🎉 HOÀN TẤT ĐÁNH GIÁ!
📊 Độ chính xác (Accuracy): 5/10 (50.00%)
📁 Báo cáo chi tiết đã được lưu tại: /kaggle/working/rag_evaluation_report.csv


Actually, the accuracy must be 7/10. Its because in the other 2 questions. The model forget to convert units (Example: Model show 0.004156 N, but the ground truth is 4.16*10^-3 N. Also, the differences between the value of pi, the model was confused about $\pi^2 = 10$ or $\pi \approx 3.14159$.

Test new architecture

In [7]:
# ============================================================================
# KAGGLE CELL v3: Physics Reasoner Hybrid + Self-Consistency + Unit-aware eval
#   NEW A: SELF-CONSISTENCY  - sample N=5 reasoning attempts, majority vote
#   NEW B: UNIT-AWARE MATCHING - count SI unit-scale matches (H<->mH, C<->nC...)
#   NEW C: CODE SANITIZER     - fix Greek-letter / spaced variable names before exec
# ============================================================================
import os, re, math, json, signal, random, statistics
from collections import Counter
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

random.seed(42)
N_SAMPLES = 5          # self-consistency samples (reduce to 3 if too slow under 60s)
SC_TEMPERATURE = 0.7   # temperature for the diverse reasoning samples

# ----------------------------------------------------------------------------
# 0. PATHS + CHROMA
# ----------------------------------------------------------------------------
chroma_client = chromadb.PersistentClient(path="/kaggle/working/rag_database")
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_collection(
    name="physics_golden_knowledge", embedding_function=sentence_transformer_ef)

TEST_FILE   = "/kaggle/input/datasets/quanghuy225/physics/Physics_Problems_Text_Only.csv"
GOLDEN_FILE = "/kaggle/input/datasets/quanghuy225/finalphysic/golden_rag_data_physic_v2.csv"
OUTPUT_REPORT_FILE = "/kaggle/working/rag_evaluation_report_v3.csv"

# ----------------------------------------------------------------------------
# 1. LLM CALL
# ----------------------------------------------------------------------------
import ollama
def call_llm(prompt, temperature=0.1):
    resp = ollama.chat(model="qwen2-math:7b",
                       messages=[{"role": "user", "content": prompt}],
                       options={"temperature": temperature})
    return resp["message"]["content"]

# ----------------------------------------------------------------------------
# 2. CHECKLISTS (same as v2)
# ----------------------------------------------------------------------------
UNIVERSAL_CHECKLIST = """
UNIVERSAL CHECKS (apply to EVERY problem; fix the solution if any fails):
[U1] UNITS: all quantities converted to SI before computing? (cm->m, uF->F, uC->C, ...)
[U2] CONSTANTS: k=9e9, g=10, eps0=8.85e-12.
[U3] SCALAR vs VECTOR: if vector, fix every component's DIRECTION before summing.
[U4] SQRT DOMAIN: any expression inside sqrt() that could be negative -> model is wrong.
[U5] DENOMINATOR: any denominator that could be zero?
[U6] OUTPUT: code ends with `final_result = <computed value>` (never hardcode).
[U7] MAGNITUDE: if a field/force STRENGTH is asked, return abs(value) (no negative magnitude).
[U8] GIVEN VALUES: use the numbers from the problem; never invent placeholders. Answer = NUMBER.
[U9] VARIABLE NAMES: no spaces, no Greek letters. Use E_net, k, eps0 (not "E Net", not kappa).
[U10] FORMULA CHECK: field of a point charge is k*q/r**2 (square the distance!), not k*q/r.
""".strip()

CHECKLIST_REGISTRY = {
    "capacitor": [
        "[CAP-DISCONNECT] Source DISCONNECTED -> Q constant (U changes with C); CONNECTED -> U constant.",
        "[CAP-ENERGY] W=0.5*C*U^2=Q^2/(2C). U doubles->W x4; Q halves(C fixed)->W /4.",
    ],
    "electrostatics_field": [
        "[EF-DIRECTION] + charge field points AWAY, - charge points TOWARD. "
        "Between OPPOSITE charges -> ADD magnitudes; between LIKE charges -> SUBTRACT. Report magnitude.",
        "[EF-FORMULA] Point charge: E = k*q/r**2 (distance SQUARED).",
        "[EF-ANGLE] Two fields at angle theta: |E|=sqrt(E1**2+E2**2+2*E1*E2*cos(theta)). Equilateral->60deg.",
        "[EF-ORDER] If collinear, fix point order from distances (MA+AB=MB => A between M,B).",
        "[EF-LINE] Infinite line charge: E = lam/(2*pi*eps0*r). Use abs(lam).",
    ],
    "electrostatics_force": [
        "[FORCE-FORMULA] Coulomb force F = k*q1*q2/r**2 (distance SQUARED). Net force = vector sum.",
    ],
    "circuit_resistance": [
        "[RES-SP] Series R=R1+R2; Parallel R=R1*R2/(R1+R2). Parallel shares voltage; series shares current.",
    ],
    "circuit_power": ["[PWR-SUM] Total power = sum of individual powers for the same source."],
    "measurement_error": ["[ERR-REL] Absolute error=|measured-true|; Relative(%)=abs_err/true*100."],
    "LC_oscillation": ["[LC-ENERGY] Ideal LC: W_C+W_L=const. If W_L=W_total/2 then W_C=W_total/2."],
    "ac_resonance": [
        "[RES-FREQ] Resonance frequency f0=1/(2*pi*sqrt(L*C)). Resonance iff supplied f ~= f0. "
        "For Yes/No set final_result to 'Yes' or 'No'."],
    "induction": ["[IND-EMF] Induced emf=-L*dI/dt; appears only when current changes with time."],
}

def get_checklist(topic):
    lines=[UNIVERSAL_CHECKLIST,"",f"TOPIC-SPECIFIC CHECKS ({topic}):"]
    items=CHECKLIST_REGISTRY.get(topic,[])
    lines+=items if items else ["(none - rely on universal checks)"]
    return "\n".join(lines)

# ----------------------------------------------------------------------------
# 3. TOPIC ROUTING (v2 logic)
# ----------------------------------------------------------------------------
VALID_TOPICS=set(CHECKLIST_REGISTRY.keys())
TOPIC_SYNONYMS={'capacitance':'capacitor','capacitor_energy':'capacitor','ac':'ac_resonance',
    'resonance':'ac_resonance','rlc':'ac_resonance','lc':'LC_oscillation','oscillation':'LC_oscillation',
    'field':'electrostatics_field','electric_field':'electrostatics_field','coulomb':'electrostatics_force',
    'force':'electrostatics_force','resistance':'circuit_resistance','resistor':'circuit_resistance',
    'power':'circuit_power','emf':'induction','inductance':'induction','error':'measurement_error',
    'uncertainty':'measurement_error'}
def normalize_topic(raw):
    if not raw: return None
    t=raw.strip().lower().replace(' ','_').replace('-','_')
    if t in VALID_TOPICS: return t
    if t in TOPIC_SYNONYMS: return TOPIC_SYNONYMS[t]
    for k,v in TOPIC_SYNONYMS.items():
        if k in t: return v
    return None
TOPIC_KEYWORDS=[('ac_resonance',['resonance','rlc','resonant']),
    ('measurement_error',['absolute error','relative error','percent error','uncertainty','measured value']),
    ('LC_oscillation',['lc circuit','oscillation','natural period','electromagnetic oscillation']),
    ('induction',['induced','inductance','self-induction','emf','solenoid','magnetic flux']),
    ('capacitor',['capacitor','capacitance','farad','charge stored','dielectric','plates']),
    ('circuit_power',['power','watt','consumes','dissipat']),
    ('circuit_resistance',['resistor','resistance','ohm','series','parallel','ammeter','current through']),
    ('electrostatics_force',['force on','coulomb force','net force','repuls','attract']),
    ('electrostatics_field',['electric field','field strength','field intensity','v/m','point charge','linear charge'])]
def classify_topic_keywords(question):
    q=question.lower()
    for topic,kws in TOPIC_KEYWORDS:
        if any(kw in q for kw in kws): return topic
    return 'electrostatics_field'
def resolve_topic(llm_topic, question):
    t=normalize_topic(llm_topic)
    return t if t in VALID_TOPICS else classify_topic_keywords(question)

# ----------------------------------------------------------------------------
# 4. STAGE 1: ANALYSIS (run ONCE; not resampled - it is classification-like)
# ----------------------------------------------------------------------------
ANALYSIS_CHECKLIST="""
ANALYSIS CHECKS:
[A1] Capacitor: source CONNECTED or DISCONNECTED?
[A2] Medium? (vacuum/air: k=9e9; dielectric: divide by eps_r)
[A3] Sign of each charge?
[A4] Where is the asked point vs the sources? If collinear, fix the order.
[A5] Scalar or vector? Is a MAGNITUDE requested?
""".strip()
VALID_TOPICS_STR=", ".join(sorted(VALID_TOPICS))
def run_analysis(question):
    prompt=f"""You are a physics expert. Analyze the problem BEFORE solving it. Do NOT compute yet.

PROBLEM:
{question}

Output exactly:
GIVEN: <known quantities with units>
UNKNOWN: <what is asked, with expected unit>
TOPIC: <choose EXACTLY ONE, copy verbatim: {VALID_TOPICS_STR}>
CONDITIONS: <source connected/disconnected, medium, charge signs, geometry>

Apply this checklist and revise if needed:
{ANALYSIS_CHECKLIST}

FINAL_ANALYSIS: <corrected analysis>
TOPIC_TAG: <repeat the chosen TOPIC value alone>
"""
    out=call_llm(prompt,0.1)
    m=re.search(r'FINAL_ANALYSIS:\s*(.+?)(?:TOPIC_TAG:|$)',out,re.DOTALL)
    analysis=m.group(1).strip() if m else out.strip()
    t=re.search(r'TOPIC_TAG:\s*([A-Za-z_ \-]+)',out) or re.search(r'TOPIC:\s*([A-Za-z_ \-]+)',out)
    return analysis, resolve_topic(t.group(1).strip() if t else "", question)

# ----------------------------------------------------------------------------
# 5. STAGE 2: RAG RETRIEVE (run ONCE)
# ----------------------------------------------------------------------------
def rag_retrieve_fn(query_text, k=2):
    res=collection.query(query_texts=[query_text], n_results=k)
    docs=res.get("documents",[[]])[0]; metas=res.get("metadatas",[[]])[0]
    blocks=[]
    for d,m in zip(docs,metas):
        code=(m or {}).get("golden_code",""); ans=(m or {}).get("answer","")
        blocks.append(f"Q: {d}\nSolution code:\n{code}\nAnswer: {ans}")
    return "\n---\n".join(blocks) if blocks else "(no similar example found)"

# ----------------------------------------------------------------------------
# 6. STAGE 3: GUIDED REASONING (resampled for self-consistency)
# ----------------------------------------------------------------------------
def build_reasoning_prompt(question, analysis, topic, examples):
    return f"""You are a physics expert. You MUST produce BOTH a Chain-of-Thought and code.

PROBLEM:
{question}

ANALYSIS:
{analysis}

SIMILAR SOLVED EXAMPLES (reference only; adapt, do not copy blindly):
{examples}

Rules:
- Use SI units. Constants: k=9e9, g=10, eps0=8.85e-12.
- The COT section is MANDATORY: at least 3 numbered steps.
- The code MUST end with `final_result = <computed value>` (never hardcode).
- Variable names: NO spaces, NO Greek letters.
- Point-charge field/force uses r**2 (squared distance).
- If a magnitude is asked, return abs(value); the answer must be a NUMBER.
- For Yes/No questions, set final_result to "Yes" or "No".
- Apply this checklist first:

{get_checklist(topic)}

Output format:
COT:
1. <step>
2. <step>
3. <step>
CODE:
```python
<code ending with final_result = ...>
```
"""

def extract_code(o):
    m=re.search(r'```python\s*(.*?)```',o,re.DOTALL)
    if m: return m.group(1).strip()
    m=re.search(r'CODE:\s*(.+)',o,re.DOTALL)
    return m.group(1).strip() if m else ""
def extract_cot(o, code):
    m=re.search(r'COT:\s*(.*?)(?:CODE:|```)',o,re.DOTALL)
    if m and m.group(1).strip(): return m.group(1).strip()
    comments=[l.strip('# ').strip() for l in (code or '').split('\n') if l.strip().startswith('#') and len(l.strip())>2]
    return '\n'.join(f"Step {i+1}: {c}" for i,c in enumerate(comments)) if comments else ""

# ----------------------------------------------------------------------------
# NEW C: CODE SANITIZER (deterministic - fix Greek + spaced variable names)
# ----------------------------------------------------------------------------
GREEK={'κ':'k','π':'pi','θ':'theta','ε':'eps','λ':'lam','ρ':'rho','μ':'mu','ω':'omega',
       'α':'alpha','β':'beta','Δ':'d','σ':'sigma','φ':'phi','Ω':'Ohm','τ':'tau','γ':'gamma'}
def sanitize_code(code):
    if not code: return code
    for g,r in GREEK.items(): code=code.replace(g,r)
    for nm in set(re.findall(r'\b([A-Za-z]\w* +\w+) *=(?!=)', code)):
        code=code.replace(nm, nm.replace(' ','_'))
    return code

# ----------------------------------------------------------------------------
# 7. SANDBOX + SYNTAX-AWARE REPAIR
# ----------------------------------------------------------------------------
class TimeoutException(Exception): pass
def _handler(s,f): raise TimeoutException()
def execute_sandbox(code, timeout=5):
    if not code: return None,"empty_code"
    code=sanitize_code(code)          # NEW C: sanitize before exec
    env={'math':math,'pi':math.pi,'sqrt':math.sqrt,'k':9e9,'g':10.0,'eps0':8.85e-12,'e':1.6e-19,'c':3e8}
    try:
        signal.signal(signal.SIGALRM,_handler); signal.alarm(timeout)
        exec(code,{'__builtins__':__builtins__},env); signal.alarm(0)
        if 'final_result' not in env: return None,"no_final_result"
        return env['final_result'],None
    except TimeoutException: return None,"timeout"
    except SyntaxError as ex: signal.alarm(0); return None,f"SyntaxError: {ex}"
    except ValueError as ex: signal.alarm(0); return None,f"ValueError: {ex} (likely sqrt of negative)"
    except Exception as ex: signal.alarm(0); return None,f"{type(ex).__name__}: {ex}"

def build_repair_prompt(question, analysis, topic, code, err):
    hint=""
    if "Syntax" in err or "empty" in err or "NameError" in err:
        hint="\nIMPORTANT: syntax/name problem. Use valid Python identifiers (no spaces, no Greek)."
    return f"""Your previous code FAILED: {err}{hint}

PROBLEM:
{question}
ANALYSIS:
{analysis}
BROKEN CODE:
```python
{code}
```
Re-examine with this checklist:
{get_checklist(topic)}
Output corrected code only:
```python
<fixed code ending with final_result = ...>
```
"""

# ----------------------------------------------------------------------------
# NEW A: SELF-CONSISTENCY pipeline
# ----------------------------------------------------------------------------
def solve_once(question, analysis, topic, examples, temperature, max_repair=1):
    out=call_llm(build_reasoning_prompt(question,analysis,topic,examples), temperature)
    code=extract_code(out); cot=extract_cot(out,code)
    result,err=execute_sandbox(code)
    rep=0
    while err and rep<max_repair:
        out=call_llm(build_repair_prompt(question,analysis,topic,code,err),0.2)
        code=extract_code(out)
        if not cot: cot=extract_cot(out,code)
        result,err=execute_sandbox(code); rep+=1
    return result, err, code, cot

def vote_answers(answers):
    answers=[a for a in answers if a is not None]
    if not answers: return None,0.0
    numeric=[]; strings=[]
    for a in answers:
        n=parse_num(a)
        (numeric if n is not None else strings).append(n if n is not None else str(a).strip().lower())
    if len(numeric)>=len(strings) and numeric:
        clusters=[]
        for v in numeric:
            for c in clusters:
                if (v==0 and c[0]==0) or (c[0]!=0 and math.isclose(v,c[0],rel_tol=2e-2)):
                    c.append(v); break
            else: clusters.append([v])
        best=max(clusters,key=len)
        return statistics.median(best), len(best)/len(answers)
    if strings:
        top,cnt=Counter(strings).most_common(1)[0]
        return top, cnt/len(answers)
    return None,0.0

def solve_physics_sc(question, n=N_SAMPLES):
    analysis,topic=run_analysis(question)              # once
    examples=rag_retrieve_fn(f"{question}\n{analysis}")# once
    cands, codes, cots = [], [], []
    for i in range(n):
        temp = 0.1 if i==0 else SC_TEMPERATURE          # 1st greedy, rest diverse
        result,err,code,cot=solve_once(question,analysis,topic,examples,temp)
        if err is None:
            cands.append(_fmt(result)); codes.append(code); cots.append(cot)
    voted,conf=vote_answers(cands)
    # pick the code/cot of a sample matching the voted answer (for explanation)
    best_code, best_cot = "", ""
    for a,cd,ct in zip(cands,codes,cots):
        if str(a)==str(voted) or (parse_num(a) is not None and parse_num(str(voted)) is not None
                                  and parse_num(a)!=0 and math.isclose(parse_num(a),parse_num(str(voted)),rel_tol=2e-2)):
            best_code,best_cot=cd,ct; break
    return {"answer": voted if voted is not None else "ERROR",
            "topic":topic,"confidence":round(conf,2),"n_valid":len(cands),
            "code":best_code,"cot":best_cot,"analysis":analysis}

def _fmt(v):
    if isinstance(v,float):
        if abs(v)>=1e4 or (abs(v)<1e-3 and v!=0): return f"{v:.4e}"
        return f"{v:g}"
    return str(v)

# ----------------------------------------------------------------------------
# NEW B: UNIT-AWARE MATCHING
# ----------------------------------------------------------------------------
def parse_num(s):
    s=str(s).strip()
    for a,b in {'⁰':'0','¹':'1','²':'2','³':'3','⁴':'4','⁵':'5','⁶':'6','⁷':'7','⁸':'8','⁹':'9','⁻':'-'}.items(): s=s.replace(a,b)
    s=s.replace('{','').replace('}','').replace(' ','')
    m=re.match(r'^(-?\d+\.?\d*)[\.\*xX×e]+10\^?(-?\d+)$',s)
    if m: return float(m.group(1))*10**int(m.group(2))
    m=re.match(r'^10\^?(-?\d+)$',s)
    if m: return 10**int(m.group(1))
    try: return float(s)
    except: return None

SI_SCALES={2,3,6,9,12}  # centi, milli/kilo, micro/mega, nano, pico
def is_correct_strict(m,g):
    if m is None or str(m)=='ERROR': return False
    mn,gn=parse_num(m),parse_num(g)
    if mn is not None and gn is not None: return math.isclose(mn,gn,rel_tol=2e-2)
    return str(m).strip().lower()==str(g).strip().lower()
def is_correct_unit(m,g):
    if is_correct_strict(m,g): return True
    mn,gn=parse_num(m),parse_num(g)
    if mn and gn and mn!=0 and gn!=0:
        log=math.log10(abs(mn/gn)); n=round(log)
        if abs(log-n)<0.02 and abs(n) in SI_SCALES: return True
    return False

# ----------------------------------------------------------------------------
# 8. TEST SET (same 40)
# ----------------------------------------------------------------------------
test=pd.read_csv(TEST_FILE); golden=pd.read_csv(GOLDEN_FILE)
test['prefix']=test['id'].astype(str).str.extract(r'^([A-Za-z]+)')
golden_ids=set(golden['id'].astype(str))
test=test[test['prefix']!='QA']
unseen=test[~test['id'].astype(str).isin(golden_ids)]
PREFIXES=['CH','NL','LD','DDT','TD','THCB','DT']
samples=[unseen[unseen['prefix']==p].sample(n=min(5,len(unseen[unseen['prefix']==p])),random_state=42) for p in PREFIXES]
sample_df=pd.concat(samples)[['id','question','answer','unit','prefix']]
chlt_new=pd.DataFrame([
 {"id":"CHLT901","prefix":"CHLT","unit":"-","answer":"Yes","question":"An RLC series circuit has R=30 ohm, L=0.6 H, and C=15 uF. When supplied with an AC frequency of 53.0 Hz, does resonance occur?"},
 {"id":"CHLT902","prefix":"CHLT","unit":"-","answer":"No","question":"A series RLC circuit consists of R=45 ohm, L=0.5 H, and C=30 uF. Is the circuit in resonance at a frequency of 35.0 Hz?"},
 {"id":"CHLT903","prefix":"CHLT","unit":"-","answer":"Yes","question":"For an RLC series circuit with R=55 ohm, L=0.9 H, and C=8 uF, does resonance occur at an operating frequency of 59.3 Hz?"},
 {"id":"CHLT904","prefix":"CHLT","unit":"-","answer":"No","question":"Given an RLC series circuit with R=25 ohm, L=0.3 H, and C=50 uF, will resonance occur if the AC frequency is 60.0 Hz?"},
 {"id":"CHLT905","prefix":"CHLT","unit":"-","answer":"Yes","question":"An RLC series circuit has R=70 ohm, L=1.2 H, and C=3 uF. When the source supplies a frequency of 83.9 Hz, is it in resonance?"},
])
sample_df=pd.concat([sample_df,chlt_new],ignore_index=True)
print(f"Test set: {len(sample_df)} questions | N_SAMPLES={N_SAMPLES}")

# ----------------------------------------------------------------------------
# 9. RUN
# ----------------------------------------------------------------------------
rows=[]
for _,q in sample_df.iterrows():
    try: out=solve_physics_sc(q['question'])
    except Exception as ex:
        out={"answer":"ERROR","topic":"-","confidence":0,"n_valid":0,"code":"","cot":"","analysis":""}
    strict=is_correct_strict(out['answer'],q['answer'])
    unit  =is_correct_unit(out['answer'],q['answer'])
    rows.append({"id":q['id'],"prefix":q['prefix'],"question":q['question'][:80],
                 "gt_answer":q['answer'],"model_answer":out['answer'],
                 "correct_strict":strict,"correct_unit":unit,
                 "confidence":out['confidence'],"n_valid":out['n_valid'],
                 "topic":out['topic'],"cot":out['cot'],"code":out['code']})
    flag="OK " if unit else "NO "
    print(f"[{flag}] {q['id']:9} gt={str(q['answer'])[:10]:10} model={str(out['answer'])[:12]:12} "
          f"conf={out['confidence']} valid={out['n_valid']}/{N_SAMPLES}")

report=pd.DataFrame(rows)
report.to_csv(OUTPUT_REPORT_FILE,index=False)

# ----------------------------------------------------------------------------
# 10. SUMMARY (strict vs unit-aware)
# ----------------------------------------------------------------------------
print("\n"+"="*60)
print(f"STRICT accuracy     : {report['correct_strict'].sum()}/{len(report)} ({100*report['correct_strict'].mean():.1f}%)")
print(f"UNIT-AWARE accuracy : {report['correct_unit'].sum()}/{len(report)} ({100*report['correct_unit'].mean():.1f}%)")
print(f"  (gap = answers correct in physics but penalized by unit scale only)")
print("\nUnit-aware accuracy by prefix:")
print(report.groupby('prefix')['correct_unit'].agg(['sum','count']))
print(f"\nMean confidence (vote agreement): {report['confidence'].mean():.2f}")
print(f"Questions with all {N_SAMPLES} samples failing: {(report['n_valid']==0).sum()}")
print(f"\nReport saved: {OUTPUT_REPORT_FILE}")

Test set: 40 questions | N_SAMPLES=5


time=2026-05-23T13:38:37.517Z level=INFO source=server.go:433 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 41945"
llama_model_loader: loaded meta data with 28 key-value pairs and 339 tensors from /root/.ollama/models/blobs/sha256-857e2f21d3ffef28100d6799ae3fc8d5c9125d5434a041b6a741dd123ba2b0fa (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2 Math 7B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2-Math
llama_model_loader: - kv   5:  

[GIN] 2026/05/23 - 13:39:54 | 200 |         1m17s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:01 | 200 |  6.563730569s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:05 | 200 |  4.089911814s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:08 | 200 |  2.985405658s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:13 | 200 |  5.739795371s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:18 | 200 |  5.118952338s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:24 | 200 |  5.053983246s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:26 | 200 |  2.514911196s |       127.0.0.1 | POST     "/api/chat"
[NO ] CH033     gt=75.03      model=no           conf=0.75 valid=4/5
[GIN] 2026/05/23 - 13:40:34 | 200 |  8.073341329s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13:40:53 | 200 | 18.945782201s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2026/05/23 - 13

time=2026-05-23T14:34:32.581Z level=INFO source=server.go:433 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 35365"
time=2026-05-23T14:34:33.246Z level=INFO source=server.go:433 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 43913"


The performance is not quite good, isnt it? So, in the next phase, we are going to add all cleaned questions to the database and transfering to fine-tune model. All raw records will be handled by codex so that we can make a verified_data_expaned. 